# 10 · Reconcile and publish a Silver release

Run this notebook from the `silver/` folder with the existing project kernel. It selects and verifies Bronze manifests, then rebuilds the required prior stages in memory. Reuse the same `silver_settings.json` to review a consistent selection across notebooks.

In [ ]:
from pathlib import Path

from IPython.display import display
import pandas as pd

import silver_notebook
from silver_notebook import booking_batch_counts, build, frame, input_frame, seed_config, settings_from_file

settings_path = Path(silver_notebook.__file__).resolve().with_name("silver_settings.json")
settings = settings_from_file(settings_path)
config = seed_config(settings_path, settings)
model = build("validate", {key: value for key, value in settings.items() if key != "seed_config"})
batch_counts = booking_batch_counts(model, config) if config is not None else []
print("Silver settings:", settings_path)
if batch_counts:
    display(pd.DataFrame(batch_counts))
display(input_frame(model))

Inspect row counts against Bronze, voucher balances, invoice-to-GL checks, calculated AR and POS controls. PASS and FAIL are evaluated comparisons. NOT_COVERED means the selected Bronze data lacks comparable evidence for that check. Resolve failed controls and error-level exceptions before publishing. Review other exceptions with the source manifest and record key available in lineage.

In [ ]:
controls = pd.DataFrame(model.controls)
display(controls)
if not controls.empty:
    display(controls.groupby("status").size().rename("count"))
display(pd.DataFrame(model.calculated_ar))
display(pd.DataFrame(model.exceptions))
display(pd.DataFrame(model.lineage).head(20))
display(pd.DataFrame(list(model.coverage.items()), columns=["area", "coverage"]))

The MinIO target writes Delta tables under one release prefix. Each table has a Delta transaction log and Parquet data files when it contains data. The release manifest is committed last and pins each table version. Paste the exact URI printed here into Gold settings. The local target remains a legacy JSONL export.

In [ ]:
from silver_notebook import publish, publish_to_minio

PUBLISH_TARGET = "preview"  # Set to "minio" after reviewing the controls.
SILVER_BUCKET = "otc-silver"

if PUBLISH_TARGET in {"minio", "local"} and settings_from_file(settings_path) != settings:
    raise ValueError("Silver settings changed after the build. Rerun the build cell before publishing.")
if PUBLISH_TARGET in {"minio", "local"} and seed_config(settings_path, settings) != config:
    raise ValueError("Seed config changed after the build. Rerun the build cell before publishing.")
if PUBLISH_TARGET in {"minio", "local"} and any(row["found"] != row["expected"] for row in batch_counts):
    raise ValueError("Seed booking counts differ. Review the batch results and Bronze manifest pins above.")

if PUBLISH_TARGET == "minio":
    print(publish_to_minio(model, bucket=SILVER_BUCKET))
elif PUBLISH_TARGET == "local":
    print((publish(model).resolve() / "manifest.json").as_posix())
elif PUBLISH_TARGET == "preview":
    print('Preview complete. Set PUBLISH_TARGET = "minio" to publish the validated release.')
else:
    raise ValueError('Use "preview", "minio", or "local".')

In [ ]:
import json
from pathlib import Path

import silver_notebook as silver
from bronze_review import open_bronze

settings_path = Path(silver.__file__).resolve().with_name("silver_settings.json")
settings = silver.settings_from_file(settings_path)
batch = settings["batch_id"].strip().upper()
expected = settings["expected_bookings"]
bronze = open_bronze()

def manifest_keys(prefix):
    return [
        key for key in bronze.objects
        if key.startswith(prefix) and key.endswith("/manifest.json")
    ]

def newest_snapshot(keys):
    if not keys:
        raise ValueError("No matching Bronze snapshots. Check the Airflow runs.")
    return max(
        keys,
        key=lambda key: (json.loads(bronze.read(key))["published_at"], key),
    )

def pos_key(erp_key):
    return "batch/parent-pos/" + erp_key.removeprefix("batch/erpnext/")

# Use a REST run that produced both the ERPNext and parent POS manifests.
erp = newest_snapshot([
    key for key in manifest_keys("batch/erpnext/")
    if pos_key(key) in bronze.objects
])
dealer = newest_snapshot(manifest_keys("batch/dealer-sql/"))

export_prefix = f"files/otc-inbound/dealer/exports/dealer-export-{batch.lower()}/"
exports = manifest_keys(export_prefix)
if len(exports) != 1:
    raise ValueError(f"Expected one dealer export under {export_prefix}, found {len(exports)}.")

settings["manifests"] = {
    **(settings.get("manifests") or {}),
    "erpnext": erp,
    "parent-pos": pos_key(erp),
    "dealer-sql": dealer,
    "dealer_export": exports[0],
}

model = silver.build("orders", settings, bronze=bronze)
orders = silver.frame(model, "sales_order")
refs = orders["order_reference"].fillna("").astype(str)
found = {
    "parent": int(refs.str.startswith(f"P-{batch}-").sum()),
    "dealer": int(refs.str.startswith(f"D-{batch}-").sum()),
}
if found != expected:
    raise ValueError(f"{batch}: found {found}, expected {expected}. Check the latest Bronze runs.")

print(json.dumps(settings, indent=2))